In [2]:
// Parameters — BATCH_MODE = "true" (papermill) : jamais lue

# Complexity-03e — Barrières d'exposants et word RAM : lire une formalisation à la machine

**Série Complexity — accrétion de 03** · Public : **Recherche** · Hommage : **Josh Alman & Virginia Vassilevska Williams** (*Truly Subquadratic 3SUM and Truly Subcubic APSP*, octobre 2026), et le projet [anthropics/formal-math · 3sum-apsp](https://github.com/anthropics/formal-math/tree/main/3sum-apsp) qui les formalise.

**Navigation** : [Index de la série](README.md) | [Version Python](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) | [Version C#](Complexity-03e-Barrieres-Exposants-WordRAM-CSharp.ipynb) | [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) | [03b — Hartmanis & Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) | [03c — Zoo et oracles](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) | [03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) | [07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) | [03f — la chaîne des réductions](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb)

> **Bandeau d'honnêteté.** Ce carnet lit une formalisation **publiée par les auteurs** et n'exécute **aucun** de leurs programmes. Les cinq claims sont des **existences de programme** sur un word RAM défini dans le fichier même qui les énonce ; leurs constantes de temps ne sont pas optimisées (« large, and for most theorems no value is given » — README du projet). La mesure de ce carnet (le contrôle quadratique) est un **témoin du point de départ** : elle ne confirme ni ne réfute la borne 1,9992. Aucun `lake build` n'est lancé ici.

## 0. Pourquoi ce carnet

Le [07c](Complexity-07c-LeBilanMathocalypse-Python.ipynb) a raconté le contrepoint Williams–Alman du lundi 5 octobre : 3SUM en O(n^1.9992), APSP en O(n^2.9995), SETH intacte. Le [03d](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) a lu la réfutation annoncée de l'ETH par les mêmes vagues d'octobre. Ce carnet prend l'autre bout de la lorgnette : pas le récit, pas l'énoncé — **le modèle de calcul**.

La formalisation [`3sum-apsp`](https://github.com/anthropics/formal-math/tree/main/3sum-apsp) énonce ses cinq claims dans **un seul fichier qui n'importe rien** — `EndStatement.lean` — et le README du projet pose la règle du jeu : « For the five claims this one file is all that has to be read. » Ce fichier définit sa propre machine (un word RAM de neuf instructions), sa propre sémantique de « résolu en temps O(n^r) », et cinq propositions. C'est une réponse **externe** au trou que la série documente depuis 03b : Mathlib n'a pas de couche `Computability/Complexity`, pas de `TIME(f)` — ici, le trou est comblé hors Mathlib, dans le noyau Lean seul.

Ce carnet fait trois choses : (1) reproduire les cinq claims avec leurs exposants **tels qu'imprimés** ; (2) **réimplémenter la machine en Rust** et la vérifier sur un témoin fermé — une instance dont le compte de pas est calculé à la main, cellule par cellule ; (3) mesurer le point de départ — le 3SUM quadratique — en disant explicitement pourquoi la borne 1,9992 ne se mesure pas aujourd'hui.

In [3]:
let seed: u32 = 20261028; // tout le carnet est rejouable a l'identique
println!("Pret : std seul, aucun import. SEED = {seed}.");

Pret : std seul, aucun import. SEED = 20261028.


## 1. Les cinq claims, tels qu'énoncés

Source : [`EndStatement.lean`](https://github.com/anthropics/formal-math/blob/main/3sum-apsp/EndStatement.lean) (lignes 76–137) et la table du [README](https://github.com/anthropics/formal-math/blob/main/3sum-apsp/README.md) du projet. Chaque claim est une **proposition Lean** (`def ... : Prop`) ; chacune est prouvée par un théorème du projet (`endStatement_*`) — la preuve vit hors de ce fichier, et ce carnet ne la recharge pas : il lit l'énoncé.

| Claim Lean | Papier | Problème | Borne | Théorème |
|---|---|---|---|---|
| `Theorem_19` | Theorem 19 | Exact Triangle | O(n^(3−0,0017)) | `endStatement_theorem_19` |
| `Theorem_22_3SUM` | Theorem 22 | 3SUM | O(n^1,9992) | `endStatement_theorem_22_3SUM` |
| `Theorem_22_MinPlus` | Theorem 22 | produit (min,+) | O(n^2,99942) | `endStatement_theorem_22_MinPlus` |
| `Theorem_22_APSP` | Theorem 22 | APSP | O(n^2,99942) | `endStatement_theorem_22_APSP` |
| `Corollary_39_ZeroWeight` | Corollaire 39 | k-Clique de poids nul | O(n^(k−0,0017·floor(k/3))) | `endStatement_corollary_39_zeroWeight` |

In [4]:
// EndStatement.lean, lignes 48-51 : le noyau Lean n'a pas de puissances fractionnaires,
// donc BigO releve les deux membres a la puissance r.den.

fn pgcd(mut a: i64, mut b: i64) -> i64 {
    while b != 0 { let t = b; b = a % b; a = t; }
    a
}

fn rat_from_decimal(s: &str) -> (i64, i64) {
    let (int_part, frac_part) = match s.find('.') {
        Some(pos) => (&s[..pos], &s[pos + 1..]),
        None => (s, ""),
    };
    let digits = if frac_part.is_empty() { int_part.to_string() }
                else { format!("{}{}", int_part, frac_part) };
    let mut num: i64 = 0;
    for c in digits.chars() { num = num * 10 + (c as i64 - '0' as i64); }
    let mut den: i64 = 1;
    for _ in 0..frac_part.len() { den *= 10; }
    let g = pgcd(num, den);
    (num / g, den / g)
}

for r in ["1.9992", "2.99942"] {
    let (num, den) = rat_from_decimal(r);
    println!("r = {} = {}/{} :  BigO dit  T(n)^{} <= K * n^{}  pour n >= 2", r, num, den, den, num);
};

r = 1.9992 = 2499/1250 :  BigO dit  T(n)^1250 <= K * n^2499  pour n >= 2


r = 2.99942 = 149971/50000 :  BigO dit  T(n)^50000 <= K * n^149971  pour n >= 2


### Lecture du résultat — deux numéros par théorème, le claim prend le plus petit

Deux détails que la table seule ne montre pas, lus à la source :

- **Theorem 22 imprime deux bornes par problème** (papier, p. 36) : celle de l'algorithme passant par Theorem 5 — n^2−1/1296+o(1) ≤ O(n^1.99923) pour 3SUM, Õ(n^3−1/1944) ≤ O(n^2.99949) pour (min,+) et APSP — et celle de l'algorithme passant par Corollary 26. Le README du projet le dit : « The claims on these three problems state the numerals of the second algorithm, which are smaller » — les claims formels portent **1,9992 et 2,99942**, les bornes les plus fortes. Le résumé du papier (p. 1) arrondit l'autre algorithme : O(n^2.9995).
- **Le BigO du fichier est rationnel** : le noyau Lean n'a pas de puissances fractionnaires, alors `BigO` relève les deux membres à la puissance `r.den` — pour r = 1,9992 = 2499/1250, la borne littérale est « T(n)^1250 ≤ K·n^2499 pour n ≥ 2 ». Aucun exposant n'est arrondi au passage : ce que Lean prouve est exactement ce que la table affiche.

Et Exact Triangle : 3 − 0,0017 = 2,9983 — le papier lui-même (p. 63) décrit la formalisation comme « Theorem 19 (Exact Triangle in O(n^2.9983) time) ».

## 2. La machine : neuf instructions, un pas chacune

`EndStatement.lean` (lignes 15–46) définit la machine **dans l'énoncé même** :

- **Neuf instructions**, chacune coûte **un pas** (l'arrêt inclus) : écrire la constante 1 dans une cellule (`one`) ; additionner, soustraire, multiplier deux cellules modulo 2^W (`add`, `sub`, `mul`) ; charger et stocker **à adresse indirecte** signée (`load`, `store`) ; sauter à une position **fixée dans le texte du programme** si une cellule est négative (`bltz`) ; `accept` ; `reject`.
- **Mémoire** : cellules nommées par des entiers — négatifs compris — tenant des mots de W bits ; hors entrée, tout vaut 0 ; l'entrée se place en cellules 0, 1, 2, … et la **taille n en cellule 0 devant** (`loadWords`, `Problem.SolvedBy`).
- **Sortie du programme** : une position hors programme compte comme `reject` (`P.getD pc .reject`, ligne 33).
- **Ce que la machine n'a pas** (README : « There is no division, no shift, no bitwise operation, and no constant other than 1 ») : ni division, ni décalage, ni opération bit à bit, et **la seule constante est 1** — le zéro lui-même se fabrique par `sub x x`, un entier k par une chaîne de doubles en ~log k pas.

La cellule suivante porte ce modèle en Rust — un **port de lecture**, fidèle à la lettre de `exec` (lignes 29–42), pas une preuve : ce qui rassure ici, c'est Lean, pas Rust.

In [5]:
// Port de l'interpreteur word RAM — EndStatement.lean lignes 29-42

use std::collections::BTreeMap;

#[derive(Debug, Clone)]
enum Instr {
    One(i64),
    Add(i64, i64, i64),
    Sub(i64, i64, i64),
    Mul(i64, i64, i64),
    Load(i64, i64),
    Store(i64, i64),
    Bltz(i64, i64),
    Accept,
    Reject,
}

fn mot_non_signe(v: u64, w: u32) -> u64 {
    v & ((1u64 << w) - 1)
}

fn signe(v: u64, w: u32) -> i64 {
    // Lecture en complement a deux -- docs/MACHINE.md : « a word is read as a
    // signed number in two's complement wherever a sign matters ».
    let v = v & ((1u64 << w) - 1);
    if v >= (1u64 << (w - 1)) { v as i64 - (1i64 << w) } else { v as i64 }
}

fn exec_wordram(p: &[Instr], mem0: &BTreeMap<i64, u64>, w: u32, t_max: i64) -> Option<(bool, BTreeMap<i64, u64>, i64)> {
    // Port de exec (EndStatement.lean, lignes 29-42).
    // Chaque instruction coute un pas, l'arret inclus ; une position hors
    // programme compte comme reject (P.getD pc .reject).
    let mut mem = mem0.clone();
    let (mut pc, mut pas) = (0i64, 0i64);
    while pas < t_max {
        pas += 1;
        let instr = if (0..p.len() as i64).contains(&pc) { &p[pc as usize] } else { &Instr::Reject };
        match instr {
            Instr::One(a) => { mem.insert(*a, 1); pc += 1; }
            Instr::Add(a, b, c) => { mem.insert(*a, mot_non_signe(mem.get(b).copied().unwrap_or(0).wrapping_add(mem.get(c).copied().unwrap_or(0)), w)); pc += 1; }
            Instr::Sub(a, b, c) => { mem.insert(*a, mot_non_signe(mem.get(b).copied().unwrap_or(0).wrapping_sub(mem.get(c).copied().unwrap_or(0)), w)); pc += 1; }
            Instr::Mul(a, b, c) => { mem.insert(*a, mot_non_signe(mem.get(b).copied().unwrap_or(0).wrapping_mul(mem.get(c).copied().unwrap_or(0)), w)); pc += 1; }
            Instr::Load(a, b) => { mem.insert(*a, mem.get(&signe(mem.get(b).copied().unwrap_or(0), w)).copied().unwrap_or(0)); pc += 1; }
            Instr::Store(a, b) => { mem.insert(signe(mem.get(a).copied().unwrap_or(0), w), mem.get(b).copied().unwrap_or(0)); pc += 1; }
            Instr::Bltz(a, b) => { pc = if signe(mem.get(a).copied().unwrap_or(0), w) < 0 { *b } else { pc + 1 }; }
            Instr::Accept => { return Some((true, mem, pas)); }
            Instr::Reject => { return Some((false, mem, pas)); } // reject
        }
    }
    None
}

fn charger_mots(w: u32, ws: &[i64]) -> BTreeMap<i64, u64> {
    // loadWords (ligne 45) : les mots d'entree en cellules 0, 1, 2, ... ;
    // toute autre cellule (adresses negatives comprises) vaut 0.
    let mut m = BTreeMap::new();
    for (a, &v) in ws.iter().enumerate() {
        m.insert(a as i64, mot_non_signe(v as u64, w));
    }
    m
}

println!("Interpreteur word RAM charge : 9 instructions, 1 pas chacune, arret inclus.");

Interpreteur word RAM charge : 9 instructions, 1 pas chacune, arret inclus.


### Lecture du résultat — le port, ce qu'il garantit et ce qu'il ne garantit pas

Le port suit `exec` instruction par instruction : un pas par instruction, arrêt inclus ; l'adresse hors programme devient `reject` ; `load`/`store` lisent l'adresse comme la valeur **signée** du mot ; les arithmétiques **enroulent modulo 2^W** (complément à deux). Ce que le port ajoute et que Lean n'a pas besoin d'écrire : un budget `t_max` de pas — le `t` de `exec`, décrémenté par la récursion chez Lean, bouclé chez nous.

Ce port ne prouve rien : il sert à **exécuter** des programmes de la machine pour les voir tourner. La preuve que les cinq claims tiennent vit dans `ThreeSumApsp/` du projet, chargée par `lake build` — que ce carnet ne lance pas.

## 3. Le témoin fermé : un décideur 3SUM pour n = 3, compté à la main

Pour vérifier la machine, le carnet écrit le plus petit décideur honnête : **3SUM pour exactement trois nombres** (mise en page de `SolvedBy` : n = 3 en cellule 0, les nombres en cellules 1, 2, 3). La machine n'ayant **ni test d'égalité ni constante 0**, tester s = a+b+c = 0 se fait en deux sauts : rejeter si s < 0, rejeter si −s < 0, sinon s = 0.

| Pos. | Instruction | Effet | Pas cumulés |
|---|---|---|---|
| 0 | `add 10, 1, 2` | [10] := a + b | 1 |
| 1 | `add 10, 10, 3` | [10] := s = a+b+c | 2 |
| 2 | `sub 11, 10, 10` | [11] := 0 (la seule façon d'obtenir 0) | 3 |
| 3 | `sub 12, 11, 10` | [12] := −s | 4 |
| 4 | `bltz 10, 7` | si s < 0 → position 7 | 5 |
| 5 | `bltz 12, 7` | si −s < 0 (s > 0) → position 7 | 6 |
| 6 | `accept` | s = 0 : accepter | 7 |
| 7 | `reject` | | |

**Compte à la main, chemin par chemin.** Instance **oui** (s = 0) : les deux `bltz` ne sautent pas, `accept` est le 7ᵉ pas — **7 pas**. Instance **non, s < 0** : le premier `bltz` (5ᵉ pas) saute en 7, `reject` est le 6ᵉ pas — **6 pas**. Instance **non, s > 0** : le second `bltz` (6ᵉ pas) saute en 7, `reject` est le 7ᵉ pas — **7 pas**.

La cellule suivante demande à l'interpréteur exactement ces trois comptes — et rien d'autre ne compte : si la machine et la main divergent d'un pas, le témoin échoue.

In [6]:
// Le decideur 3SUM pour n = 3, tel que compte a la main ci-dessus.

use std::collections::BTreeMap;

#[derive(Debug, Clone)]
enum Instr {
    One(i64),
    Add(i64, i64, i64),
    Sub(i64, i64, i64),
    Mul(i64, i64, i64),
    Load(i64, i64),
    Store(i64, i64),
    Bltz(i64, i64),
    Accept,
    Reject,
}

fn mot_non_signe(v: u64, w: u32) -> u64 {
    v & ((1u64 << w) - 1)
}

fn signe(v: u64, w: u32) -> i64 {
    let v = v & ((1u64 << w) - 1);
    if v >= (1u64 << (w - 1)) { v as i64 - (1i64 << w) } else { v as i64 }
}

fn exec_wordram(p: &[Instr], mem0: &BTreeMap<i64, u64>, w: u32, t_max: i64) -> Option<(bool, BTreeMap<i64, u64>, i64)> {
    let mut mem = mem0.clone();
    let (mut pc, mut pas) = (0i64, 0i64);
    while pas < t_max {
        pas += 1;
        let instr = if (0..p.len() as i64).contains(&pc) { &p[pc as usize] } else { &Instr::Reject };
        match instr {
            Instr::One(a) => { mem.insert(*a, 1); pc += 1; }
            Instr::Add(a, b, c) => { mem.insert(*a, mot_non_signe(mem.get(b).copied().unwrap_or(0).wrapping_add(mem.get(c).copied().unwrap_or(0)), w)); pc += 1; }
            Instr::Sub(a, b, c) => { mem.insert(*a, mot_non_signe(mem.get(b).copied().unwrap_or(0).wrapping_sub(mem.get(c).copied().unwrap_or(0)), w)); pc += 1; }
            Instr::Mul(a, b, c) => { mem.insert(*a, mot_non_signe(mem.get(b).copied().unwrap_or(0).wrapping_mul(mem.get(c).copied().unwrap_or(0)), w)); pc += 1; }
            Instr::Load(a, b) => { mem.insert(*a, mem.get(&signe(mem.get(b).copied().unwrap_or(0), w)).copied().unwrap_or(0)); pc += 1; }
            Instr::Store(a, b) => { mem.insert(signe(mem.get(a).copied().unwrap_or(0), w), mem.get(b).copied().unwrap_or(0)); pc += 1; }
            Instr::Bltz(a, b) => { pc = if signe(mem.get(a).copied().unwrap_or(0), w) < 0 { *b } else { pc + 1 }; }
            Instr::Accept => { return Some((true, mem, pas)); }
            Instr::Reject => { return Some((false, mem, pas)); } // reject
        }
    }
    None
}

fn charger_mots(w: u32, ws: &[i64]) -> BTreeMap<i64, u64> {
    let mut m = BTreeMap::new();
    for (a, &v) in ws.iter().enumerate() {
        m.insert(a as i64, mot_non_signe(v as u64, w));
    }
    m
}

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

fn py_list_i64(xs: &[i64]) -> String {
    let items: Vec<String> = xs.iter().map(|x| x.to_string()).collect();
    format!("[{}]", items.join(", "))
}

let P_3SUM = vec![
    Instr::Add(10, 1, 2),     // 0 : [10] := a + b
    Instr::Add(10, 10, 3),    // 1 : [10] := s = a + b + c
    Instr::Sub(11, 10, 10),   // 2 : [11] := 0
    Instr::Sub(12, 11, 10),   // 3 : [12] := 0 - s
    Instr::Bltz(10, 7),       // 4 : si s < 0  -> reject
    Instr::Bltz(12, 7),       // 5 : si -s < 0 -> reject
    Instr::Accept,            // 6 : s = 0
    Instr::Reject,            // 7
];

fn temoin(p: &[Instr], nombres: &[i64], verdict_attendu: bool, pas_attendu: i64) -> bool {
    let m0 = charger_mots(16, &[3, nombres[0], nombres[1], nombres[2]]);
    let (verdict, _mem, pas) = exec_wordram(p, &m0, 16, 100).unwrap();
    let ok = (verdict == verdict_attendu) && (pas == pas_attendu);
    println!("  {:<16} -> verdict={:<5} pas={} (attendu : verdict={:<5}, pas={})  [{}]",
        py_list_i64(nombres), py_bool(verdict), pas, py_bool(verdict_attendu), pas_attendu,
        if ok { "OK" } else { "ECHEC" });
    ok
}

println!("Trois cas canoniques, un par chemin du programme :");
let cas: Vec<(&[i64], bool, i64)> = vec![
    (&[-5, 2, 3][..],   true,  7),   // s = 0 : chemin accept
    (&[-3, -2, -1][..], false, 6),   // s < 0 : saut au 5e pas
    (&[ 1,  2,  3][..], false, 7),   // s > 0 : saut au 6e pas
];
let mut tous_ok = true;
for (nb, vd, pp) in &cas {
    if !temoin(&P_3SUM, nb, *vd, *pp) { tous_ok = false; }
}

// Batterie seedee : le verdict de la machine contre la definition ThreeSum de la
// source (ligne 92 : trois positions DEUX A DEUX DISTINCTES ; a n = 3, l'unique triple).
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {                     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits, deux mots
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 32 ici : UN mot entier, tronque
        (self.next_u32() >> (32 - k)) as u64
    }
    fn randbelow(&mut self, n: u64) -> u64 {    // _randbelow : rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn sample2_of_3(&mut self) -> (u64, u64) {  // random.sample(range(3), 2) : chemin pool (n = 3 <= setsize 21)
        let mut pool = [0u64, 1, 2];
        let j = self.randbelow(3) as usize;     // i = 0 : j = _randbelow(n - 0)
        let r0 = pool[j];
        pool[j] = pool[2];                      // pool[j] = pool[n - i - 1]
        let j = self.randbelow(2) as usize;     // i = 1 : j = _randbelow(n - 1)
        (r0, pool[j])
    }
}

let mut rng = Mt19937::new(20261028);
let mut accords = 0;
let essais = 300;
for _ in 0..essais {
    let mut nb = [0i64; 3];
    for k in 0..3 { nb[k] = rng.randint(-20, 20); }
    let u = rng.random();
    if u < 0.35 {
        let (i, j) = rng.sample2_of_3();
        nb[(3 - i - j) as usize] = -(nb[i as usize] + nb[j as usize]); // forcer des cas oui
    }
    let attendu = nb[0] + nb[1] + nb[2] == 0;
    let m0 = charger_mots(16, &[3, nb[0], nb[1], nb[2]]);
    let (verdict, _, _) = exec_wordram(&P_3SUM, &m0, 16, 100).unwrap();
    if verdict == attendu { accords += 1; }
}
println!("Batterie seedee : {}/{} accords avec la definition de la source.", accords, essais);
println!("TEMOIN FERME : {} -- le compte de pas de l'interpreteur egale le compte a la main sur les trois chemins.",
    if tous_ok && accords == essais { "CONFIRME" } else { "ECHEC" });

Trois cas canoniques, un par chemin du programme :


  [-5, 2, 3]       -> verdict=True  pas=7 (attendu : verdict=True , pas=7)  [OK]


  [-3, -2, -1]     -> verdict=False pas=6 (attendu : verdict=False, pas=6)  [OK]


  [1, 2, 3]        -> verdict=False pas=7 (attendu : verdict=False, pas=7)  [OK]


Batterie seedee : 300/300 accords avec la definition de la source.


TEMOIN FERME : CONFIRME -- le compte de pas de l'interpreteur egale le compte a la main sur les trois chemins.


### Lecture du résultat — la machine et la main tombent d'accord

Sur les trois chemins (oui ; non par s < 0 ; non par s > 0), le compte de l'interpréteur égale le compte à la main, et sur la batterie seedée le verdict suit la définition `ThreeSum` de la source. Le témoin ne dit rien des cinq claims : il dit que **le modèle est bien celui qu'on a lu** — neuf instructions, un pas chacune, l'arrêt compté, le hors-programme qui rejette.

## 4. Le point de départ, mesuré — et pourquoi la borne ne se mesure pas

Le contrôle : le **3SUM quadratique** classique (tri + deux pointeurs, O(n²)) chronométré par doublement, sur des instances **sans solution** (trois nombres positifs ne somment jamais à 0) — sinon le chrono mesure « où se trouve le premier triple trouvé », pas le balayage complet ; c'est la leçon de méthode de cette section. Ce point de départ est celui que les bornes cassées devancent — **témoin du point de départ, jamais une mesure du gain annoncé**. Ce contrôle **ne réfute rien** : une pente ≈ 2 est exactement ce que la borne O(n^1.9992) elle-même prédit pour tout n atteignable, parce que (README du projet) « The constants of the time bounds are not optimized; they are large, and for most theorems no value is given » — l'écart d'exposant 2 − 1,9992 = 0,0008 vaut un facteur ~1,007 à n = 8 000, sous le bruit de n'importe quelle chronométrie. La cellule calcule ce facteur au lieu de le supposer.

In [7]:
// 3SUM quadratique : tri + deux pointeurs par ancre

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

fn trois_somme_quadratique(xs: &[i64]) -> bool {
    let mut t = xs.to_vec();
    t.sort();
    let n = t.len() as i64;
    for i in 0..(n - 2) {
        let cible = -t[i as usize];
        let (mut g, mut d) = (i + 1, n - 1);
        while g < d {
            let s = t[g as usize] + t[d as usize];
            if s == cible { return true; }
            if s < cible { g += 1; } else { d -= 1; }
        }
    }
    false
}

struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {                     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits, deux mots
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 32 ici : UN mot entier, tronque
        (self.next_u32() >> (32 - k)) as u64
    }
    fn randbelow(&mut self, n: u64) -> u64 {    // _randbelow : rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn sample2_of_3(&mut self) -> (u64, u64) {  // random.sample(range(3), 2) : chemin pool (n = 3 <= setsize 21)
        let mut pool = [0u64, 1, 2];
        let j = self.randbelow(3) as usize;     // i = 0 : j = _randbelow(n - 0)
        let r0 = pool[j];
        pool[j] = pool[2];                      // pool[j] = pool[n - i - 1]
        let j = self.randbelow(2) as usize;     // i = 1 : j = _randbelow(n - 1)
        (r0, pool[j])
    }
}

// Temoin de COUT COMPLET : instances SANS solution (trois positifs ne somment jamais a 0),
// sinon le chrono mesure « ou se trouve le premier triple », pas le balayage quadratique.
let mut rng = Mt19937::new(20261028);
let tailles: [i64; 4] = [1000, 2000, 4000, 8000];
let REPS = 3; // mediane de 3 passages : les petites tailles sont sous le bruit d'une passe unique
let _ = trois_somme_quadratique(&[1, 2, 3]); // passe a blanc : caches et allocs hors mesure

let mut mesures: Vec<(i64, f64)> = Vec::new();
for &n in &tailles {
    let mut xs: Vec<i64> = Vec::with_capacity(n as usize);
    for _ in 0..n {
        xs.push(rng.randint(1, n * n)); // tous positifs : reponse False assuree
    }
    let mut temps: Vec<f64> = Vec::new();
    let mut rep = false;
    for _ in 0..REPS {
        let dep = std::time::Instant::now();
        rep = trois_somme_quadratique(&xs);
        temps.push(dep.elapsed().as_secs_f64());
    }
    temps.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let dt = temps[REPS / 2];
    mesures.push((n, dt));
    println!("n = {:>5} : {:>8.1} ms (mediane de {}, reponse {})", n, dt * 1000.0, REPS, py_bool(rep));
}

// Pente par regression log-log globale, plus robuste que la moyenne des ratios par doublement
let ln_n: Vec<f64> = mesures.iter().map(|(n, _)| (*n as f64).ln()).collect();
let ln_t: Vec<f64> = mesures.iter().map(|(_, t)| t.ln()).collect();
let mn = ln_n.iter().sum::<f64>() / ln_n.len() as f64;
let mt = ln_t.iter().sum::<f64>() / ln_t.len() as f64;
let pente = ln_n.iter().zip(ln_t.iter()).map(|(a, b)| (a - mn) * (b - mt)).sum::<f64>()
    / ln_n.iter().map(|a| (a - mn) * (a - mn)).sum::<f64>();
let gain_8000 = 8000f64.powf(2.0 - 1.9992);
println!();
println!("Pente empirique du quadratique (regression log-log sur 4 tailles) : {:.3} -- temoin du point de depart.", pente);
println!("Ce que n^(2-1.9992) vaut a n = 8000 : un facteur {:.4} -- sous le bruit de la mesure ci-dessus.", gain_8000);

n =  1000 :      1.5 ms (mediane de 3, reponse False)


n =  2000 :      4.6 ms (mediane de 3, reponse False)


n =  4000 :     19.8 ms (mediane de 3, reponse False)


n =  8000 :     77.1 ms (mediane de 3, reponse False)


Pente empirique du quadratique (regression log-log sur 4 tailles) : 1.927 -- temoin du point de depart.


Ce que n^(2-1.9992) vaut a n = 8000 : un facteur 1.0072 -- sous le bruit de la mesure ci-dessus.


### Lecture du résultat — pente mesurée 1,927, quotient des puissances ≈ 1,007

La pente empirique du quadratique mesurée par la cellule précédente vaut **1,927** (régression log-log sur les 4 tailles, de 1,5 ms à n = 1 000 à 77,1 ms à n = 8 000) — c'est le **témoin** attendu d'un balayage quadratique. L'écart à 2 relève de la mesure, pas de l'algorithme : sans triple solution, l'ancre i fait exactement n − i − 2 pas de pointeurs, soit (n − 1)(n − 2)/2 pas au total — un compte **exactement quadratique**. Ce qui varie d'une taille à l'autre, c'est le temps par pas ; ce carnet n'en mesure pas la cause (candidats : coûts fixes du tri et de la copie, qui pèsent davantage à n = 1 000 ; caches ; résolution du chronomètre sur 1,5 ms). Le jumeau Python, lui, mesure 2,016.

À n = 8 000, n^0,0008 ≈ 1,007 : le quotient des seules puissances vaut environ 1,007, sans déterminer le rapport des temps ni leurs constantes — bien moins que l'écart entre deux exécutions de la même cellule. C'est la leçon du 07c sur le matmul en n^2,25 (crossover repoussé à ~n^10^16 si la constante vaut 100×) transposée : la victoire est **réelle en exposant** — elle existe comme théorème, et même comme programme word RAM —, **invisible en instance** tant que les constantes ne sont pas optimisées, et la source dit elle-même qu'elles ne le sont pas.

## 5. Ce que la source ne prouve pas

Quatre limites, chacune citée du [README](https://github.com/anthropics/formal-math/blob/main/3sum-apsp/README.md) du projet (sections « What is not proved » et « The five claims ») :

1. **Les entrées réelles et la section 5.1 ne sont pas prouvées** — « the running times for real inputs (Theorem 35, Lemmas 36 and 37, Corollary 38, and the half of Theorem 2 on real inputs) and those of Section 5.1 (Theorems 33 and 34) » ne sont prouvées **pour aucune machine** : l'algorithme Las Vegas O(n^1.998)/O(n^2.998) du papier (p. 4) vit sur un real RAM que cette machine n'a pas.
2. **Les constantes ne sont pas optimisées** — « The constants of the time bounds are not optimized; they are large, and for most theorems no value is given. »
3. **Les opérations absentes de la machine** — « There is no division, no shift, no bitwise operation, and no constant other than 1. » Une borne prouvée sur cette machine est donc **plus forte** que la même borne sur un word RAM standard (docs/MACHINE.md : « an upper bound on time proved for it is stronger »).
4. **Le claim est une existence de programme** — « The claims say that programs exist. The proofs describe the programs, but not everything in them is computed » (le README renvoie à docs/MACHINE.md, « How explicit the programs are »).

Et deux prudences de plus, lues au même endroit : les hypothèses que le papier « réfute » (3SUM, APSP, Exact Triangle, k-Clique) **ne sont pas définies** dans la formalisation — seuls des temps de calcul sont énoncés ; et « This README records no run of the tool » — les auteurs décrivent la procédure de vérification Comparator mais n'en publient pas d'exécution. Ce carnet, de son côté, ne lance **aucun** `lake build`.

## 6. Plus faible ici, plus généreuse là

`docs/MACHINE.md` compare la machine au word RAM standard, point par point. **Plus faible en trois points** (donc des bornes plus fortes) : pas de division, décalage ni opération bit à bit ; **la seule constante est 1** — « a constant k has to be built, in about log k steps » ; le programme ne choisit pas sa taille de mot. **Aussi généreuse en deux** : une multiplication coûte un pas, et toute cellule dont le nom tient dans un mot est adressable. Quatre gratuités supplémentaires (cellules hors entrée à 0 sans coût, cellules nommées dans le texte hors de portée des mots, pente b choisie après κ, entrées de phases gratuites) — « none of them changes an exponent ».

La taille de mot se règle ainsi : le programme est fixé **avec** un nombre b, et doit tenir sa borne à toute taille W ≥ b·(1 + ⌊log₂ n⌋) — c'est le « O(log n)-bit words » du papier (Theorem 2), rendu à la lettre dans `SolvedInTime` (lignes 67–70).

## 7. Place dans la série

- **07c** (14 octobre) écrivait : « aucun de ces résultats (UGC, L=BPL, parité ∉ QAC⁰, exposants matmul/entiers, 3SUM sous-quadratique) n'a de formalisation référencée dans ce dépôt ». Cette phrase était vraie à sa date ; le projet `3sum-apsp` la rend périmée pour le 3SUM — un renvoi daté est ajouté dans 07c par la même PR que ce carnet.
- **03d**, le carnet frère : l'autre annonce d'octobre 2026 (la réfutation de l'ETH), lue sur ses trois surfaces Lean. Ici comme là-bas, la citation croisée se mesure : le **papier cite sa formalisation** (p. 63 : « The following are formalized there: Theorem 19… Theorem 22… Corollary 39 ») et la **formalisation cite le papier** — les deux sens, contrairement au cas ETH où la cross-citation n'allait que dans un sens.
- **Le gap Mathlib** (03b) : pas de `TIME(f)` dans Mathlib — et ce projet montre la réponse des auteurs : définir machine, sémantique et bornes **dans le noyau Lean seul**, sans Mathlib dans l'énoncé (`EndStatement.lean` n'importe rien). Le prix : un BigO rationnel (§1) et une machine minimaliste (§2) ; le gain : l'énoncé entier des cinq claims tient dans un seul fichier, vérifiable ligne à ligne.

Le papier ajoute, en son propre acknowledgments (p. 63) : « The authors used Claude to help with writing, figure creation, and checking mathematical details throughout this project. They have attempted to make the entire paper accessible and well-written, and take full responsibility » — et p. 1 : « Claude also verified this paper's main results using the Lean 4 proof assistant with the Mathlib library ».

## 8. Exercices

Trois exercices, sur le modèle de la série — stubs non bloquants (C.1) : le carnet s'exécute de bout en bout même non complété.

In [8]:
// Exercice 1 — construire_constante

#[derive(Debug, Clone)]
enum Instr {
    One(i64),
    Add(i64, i64, i64),
    Sub(i64, i64, i64),
    Mul(i64, i64, i64),
    Load(i64, i64),
    Store(i64, i64),
    Bltz(i64, i64),
    Accept,
    Reject,
}

fn construire_constante(_k: i64) -> Option<Vec<Instr>> {
    // TODO etudiant : construire k par une chaine de doubles puis decomposition binaire
    // Indice : 1 -> 2 -> 4 -> ... -> 2^j, puis k en binaire
    // Etape 1 : le cas k = 2^j
    // Etape 2 : generaliser
    println!("Exercice 1 : construire_constante definie -- a completer (retour attendu : la liste d'instructions).");
    None
}

construire_constante(10);

Exercice 1 : construire_constante definie -- a completer (retour attendu : la liste d'instructions).


**Exercice 2** — traduire le contrôle de la section 4 en **pas de machine** : le décideur quadratique vu par `exec`.

In [9]:
// Exercice 2 — bornes_pas_3sum_quadratique

fn bornes_pas_3sum_quadratique(_n: i64) -> Option<String> {
    // TODO etudiant
    // Indice : boucle externe x boucle interne x cout d'une iteration traduite
    // Etape 1 : couter une iteration du two-pointer en instructions
    // Etape 2 : multiplier, puis verifier la coherence avec le temoin ferme (section 3)
    println!("Exercice 2 : bornes_pas_3sum_quadratique definie -- a completer (retour attendu : un ordre de grandeur).");
    None
}

bornes_pas_3sum_quadratique(1000);

Exercice 2 : bornes_pas_3sum_quadratique definie -- a completer (retour attendu : un ordre de grandeur).


**Exercice 3** — les deux algorithmes de Theorem 22 : mesurer l'écart que le claim laisse sur la table.

In [10]:
// Exercice 3 — ecart_deux_algorithmes

use std::collections::BTreeMap;

fn ecart_deux_algorithmes() -> Option<BTreeMap<&'static str, Vec<(f64, f64, f64)>>> {
    // TODO etudiant
    // Indice : facteur = n ** ecart, comme la section 4 pour 2 - 1.9992
    println!("Exercice 3 : ecart_deux_algorithmes definie -- a completer (retour attendu : le dict des ecarts).");
    None
}

ecart_deux_algorithmes();

Exercice 3 : ecart_deux_algorithmes definie -- a completer (retour attendu : le dict des ecarts).


## 9. Conclusion

| Geste | Où | Ce qui a été fait | Niveau |
|---|---|---|---|
| Les cinq claims | §1 | Noms Lean, items du papier, exposants tels qu'imprimés ; les deux numéros de Theorem 22 ; le BigO rationnel (2499/1250) | Lu (fichiers) |
| La machine | §2–§3 | Port Rust de `exec`/`loadWords` ; témoin fermé : compte de pas machine = compte à la main sur les trois chemins ; 300 instances seedées d'accord avec `ThreeSum` | **Mesuré** |
| Le point de départ | §4 | 3SUM quadratique chronométré (pente log-log mesurée 1,927) ; le quotient des puissances n^0,0008 (~1,007 à n = 8 000), sans mesure de la borne citée | **Mesuré** (témoin) |
| Ce que la source ne prouve pas | §5 | Quatre limites citées + deux prudences (hypothèses non définies, aucun run de Comparator publié) | Cité |
| La machine contre le standard | §6 | 3 faiblesses / 2 générosités / 4 gratuités (docs/MACHINE.md) | Cité |
| Place dans la série | §7 | Renvoi daté dans 07c ; 03d le frère ; gap Mathlib répondu hors Mathlib | Cité |

La conclusion en une phrase : **cinq claims d'existence de programme sont énoncés sur une machine de neuf instructions définie dans le fichier même qui les porte — et la machine, réimplémentée, tombe d'accord avec le compte à la main pendant que le quotient des puissances vaut 1,007 à n = 8 000, sans déterminer le rapport des temps.**

### Références (copie archivée au gisement)

- **Article** — Josh Alman & Virginia Vassilevska Williams, *Truly Subquadratic 3SUM and Truly Subcubic APSP via Triangles in Sparse Lopsided Graphs*, arXiv:2610.06783v1, 5 octobre 2026, 76 p. : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - Alman Vassilevska Williams - Truly Subquadratic 3SUM and Truly Subcubic APSP via Triangles in Sparse Lopsided Graphs (arXiv 2610.06783v1).pdf`
- **Formalisation** (lue au 28 octobre 2026) : [anthropics/formal-math · 3sum-apsp](https://github.com/anthropics/formal-math/tree/main/3sum-apsp) — `EndStatement.lean`, `README.md`, `docs/MACHINE.md`
- **Dans la série** : [03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) · [07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) · [03b — Hartmanis & Stearns et le gap Mathlib](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb)

*Issue de suivi : #23140. Carnet écrit le 28 octobre 2026 ; les faits datés (formalisation, statut des preuves) sont ceux de cette date.*

## À retenir

1. **Cinq claims d'existence sur une machine de neuf instructions.** Exact Triangle en $O(n^{3-0{,}0017})$, 3SUM en $O(n^{1{,}9992})$, produit (min,+) et APSP en $O(n^{2{,}99942})$, k-Clique de poids nul en $O(n^{k-0{,}0017\lfloor k/3 \rfloor})$ — énoncés dans le fichier même qui définit la machine qui les exécute.
2. **Le BigO rationnel relève les exposants.** Le noyau Lean n'a pas de puissances fractionnaires : $1{,}9992 = 2499/1250$ devient $T(n)^{1250} \leq K \cdot n^{2499}$ pour $n \geq 2$ (et $2{,}99942 = 149971/50000$ pareillement).
3. **La machine réimplémentée tombe d'accord avec le compte à la main.** Les trois chemins canoniques du décideur 3SUM — $[-5, 2, 3]$ accept en 7 pas, $[-3, -2, -1]$ reject en 6, $[1, 2, 3]$ reject en 7 — plus une batterie de 300 instances seedées : **300/300** accords avec la définition de la source, témoin fermé CONFIRMÉ.
4. **La borne est réelle en exposant, invisible en instance.** Le 3SUM quadratique chronométré ici monte de 1,5 ms ($n = 1\,000$) à 77,1 ms ($n = 8\,000$), pente log-log **1,927** ; le quotient des puissances $n^{0{,}0008}$ vaut **1,0072** à $n = 8\,000$ — sous le bruit de la mesure. La pente elle-même s'écarte de 2 bien plus que l'exposant en jeu : un chronomètre à ces tailles ne départage pas $n^2$ de $n^{1{,}9992}$.

## Vérifiez votre compréhension

1. Pourquoi l'interpréteur compte-t-il 7 pas pour $[-5, 2, 3]$ et 6 pour $[-3, -2, -1]$ ?
2. La pente mesurée du quadratique vaut 1,927 et la borne promise $n^{1{,}9992}$. Pourquoi cette dernière est-elle invérifiable au chronomètre aujourd'hui ?
3. Que signifie l'élévation à la puissance 1250 dans l'énoncé BigO de Lean ?

<details><summary>Réponses</summary>

1. Le premier exécute tout le chemin jusqu'à l'instruction `accept` (7 pas) ; le second saute au `reject` dès le test $s < 0$, au 5e pas (6 pas au total). Les trois chemins du programme sont exactement les trois cas canoniques comptés à la main.
2. L'écart entre $n^2$ et $n^{1{,}9992}$ est un facteur $n^{0{,}0008}$, soit 1,0072 à $n = 8\,000$ — plus petit que le bruit du chronomètre (la mesure elle-même varie de plusieurs pour cent entre passes). La pente mesurée, 1,927, tombe d'ailleurs sous 1,9992 sans que l'algorithme quadratique ait changé : son écart à 2 vient de la mesure, pas de l'exposant. Le voir demanderait des tailles inatteignables.
3. Le noyau ne connaît que des exposants entiers ; $1{,}9992 = 2499/1250$, donc l'inégalité s'écrit $T(n)^{1250} \leq K \cdot n^{2499}$ pour $n \geq 2$ — même contenu, forme à exposants entiers.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Complexity-04 — Algorithmes en ligne](Complexity-04-OnlineAlgorithms-Python.ipynb) — quitter le monde des bornes pour celui des décisions sans connaître la suite : skis, pagination, secrétaire.
- **Dans la même série** : [Complexity-03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) — le frère de ce carnet, autre annonce vérifiée à la source ; [Complexity-07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) — où ces claims prennent place dans le panorama.
- **Carnet voisin d'une autre série** : [Z3-20 — La machine de Gödel (Python)](../../SymbolicAI/SMT/Z3-API/Z3-20-Machine-de-Goedel-Python.ipynb) — une autre machine à programmes, cette fois pour démontrer plutôt que décider.
- **Lectures** : Alman et Vassilevska Williams (2026), *Truly Subquadratic 3SUM and Truly Subcubic APSP* — copie archivée au gisement partagé (chemin complet dans les Références de la Conclusion ci-dessus), où la formalisation `3sum-apsp` est aussi référencée.

> Carnet issu de l'EPIC #26534.